# 第2章 强化学习核心要素

本章介绍强化学习的核心要素，包括智能体、环境、交互模型、策略与策略评估。

## 1. 环境配置

In [ ]:
!pip install numpy matplotlib -q
import numpy as np
import random
import matplotlib.pyplot as plt
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

class GridEnvironment:
    """网格世界环境"""
    def __init__(self, n=4, stochastic=False, transition_prob=0.8):
        self.n = n
        self.state = (0, 0)
        self.terminal_state = (n-1, n-1)
        self.stochastic = stochastic
        self.transition_prob = transition_prob
        self.actions = ['up', 'down', 'left', 'right']
    
    def reset(self):
        self.state = (0, 0)
        return self.state
    
    def step(self, action):
        x, y = self.state
        if self.stochastic and np.random.random() > self.transition_prob:
            action = np.random.choice(self.actions)
        
        if action == 'up' and x > 0:
            x -= 1
        elif action == 'down' and x < self.n - 1:
            x += 1
        elif action == 'left' and y > 0:
            y -= 1
        elif action == 'right' and y < self.n - 1:
            y += 1
        
        self.state = (x, y)
        reward = 1 if self.state == self.terminal_state else 0
        done = self.state == self.terminal_state
        return self.state, reward, done

env = GridEnvironment()
print(f"环境初始化状态: {env.reset()}")
state, reward, done = env.step('right')
print(f"执行动作right后状态: {state}, 奖励: {reward}, 终止: {done}")

## 2. 编程题1: 障碍物网格世界环境

### 题目描述
修改GridEnvironment类，实现一个具有障碍物的网格世界环境：网格大小为5x5，障碍物位置为(1,1)、(2,3)、(3,1)，智能体无法进入障碍物位置（碰撞障碍物时奖励-5，且状态不发生变化），完成环��的reset()与step()方法，并测试交互过程。

In [ ]:
class GridEnvironmentWithObstacles:
    """带障碍物的网格世界环境"""
    def __init__(self, n=5):
        self.n = n
        self.obstacles = [(1,1), (2,3), (3,1)]
        self.state = (0, 0)
        self.terminal_state = (n-1, n-1)
        self.actions = ['up', 'down', 'left', 'right']
    
    def reset(self):
        self.state = (0, 0)
        return self.state
    
    def step(self, action):
        old_state = self.state
        x, y = self.state
        
        if action == 'up' and x > 0:
            x -= 1
        elif action == 'down' and x < self.n - 1:
            x += 1
        elif action == 'left' and y > 0:
            y -= 1
        elif action == 'right' and y < self.n - 1:
            y += 1
        
        new_state = (x, y)
        
        if new_state in self.obstacles:
            self.state = old_state
            reward = -5
            done = False
        else:
            self.state = new_state
            reward = 1 if self.state == self.terminal_state else 0
            done = self.state == self.terminal_state
        
        return self.state, reward, done

env = GridEnvironmentWithObstacles()
env.reset()
print("=== 障碍物网格世界环境测试 ===")
print(f"网格大小: 5x5")
print(f"障碍物位置: {env.obstacles}")
print(f"初始状态: {env.state}")

path = [('right', 1), ('right', 1), ('down', 1), ('down', 1)]
total_reward = 0
for action, expected_reward in path:
    state, reward, done = env.step(action)
    total_reward += reward
print(f"测试路径执行后状态: {state}, 总奖励: {total_reward}, 终止: {done}")

## 3. 编程题2: Softmax策略智能体

### 题目描述
实现一个基于Softmax策略的智能体，通过温度参数$\tau$控制探索与利用的平衡，编写方法选择动作，并对比不同$\tau$值（0.1、1、10）下智能体的动作选择规律，可视化不同$\tau$值对应的动作概率分布。

In [ ]:
class SoftmaxAgent:
    """Softmax策略智能体"""
    def __init__(self, n_actions=4, tau=1.0):
        self.n_actions = n_actions
        self.tau = tau
        self.actions = ['up', 'down', 'left', 'right']
        self.q_values = np.zeros(n_actions)
    
    def get_action_probs(self):
        """计算各动作的概率分布"""
        exp_q = np.exp(self.q_values / self.tau)
        probs = exp_q / exp_q.sum()
        return probs
    
    def select_action(self):
        """根据Softmax策略选择动作"""
        probs = self.get_action_probs()
        action_idx = np.random.choice(self.n_actions, p=probs)
        return self.actions[action_idx]
    
    def update_q(self, action_idx, reward, alpha=0.1):
        """更新动作价值"""
        self.q_values[action_idx] += alpha * (reward - self.q_values[action_idx])

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
tau_values = [0.1, 1, 10]
colors = ['blue', 'green', 'red']

for idx, tau in enumerate(tau_values):
    agent = SoftmaxAgent(n_actions=4, tau=tau)
    agent.q_values = np.array([3.0, 1.0, 0.5, 0.2])
    probs = agent.get_action_probs()
    
    axes[idx].bar(agent.actions, probs, color=colors[idx], alpha=0.7)
    axes[idx].set_title(f'Temperature $\\u03c4$ = {tau}')
    axes[idx].set_xlabel('动作')
    axes[idx].set_ylabel('概率')
    axes[idx].set_ylim(0, 1)

plt.suptitle('不同温度参数下的Softmax策略动作概率分布', fontsize=14)
plt.tight_layout()
plt.show()

print("结论: tau越小概率分布越集中( exploitation)，tau越大概率分布越均匀( exploitation)")

## 4. 编程题3: TD(0)策略评估

### 题目描述
结合编程题1的障碍物网格环境与编程题2的Softmax策略智能体，实现时序差分学习法（TD(0)），完成策略评估，计算动作价值函数，并可视化评估后的价值分布，分析障碍物对价值函数的影响。

In [ ]:
class TDAgent:
    """TD(0)策略评估智能体"""
    def __init__(self, env, tau=1.0, alpha=0.1, gamma=0.9):
        self.env = env
        self.tau = tau
        self.alpha = alpha
        self.gamma = gamma
        self.n_states = env.n * env.n
        self.v = np.zeros(self.n_states)
        self.actions = ['up', 'down', 'left', 'right']
    
    def state_to_idx(self, state):
        return state[0] * self.env.n + state[1]
    
    def get_action_probs(self, state):
        """Softmax策略"""
        exp_q = np.exp(np.zeros(4))
        probs = exp_q / exp_q.sum()
        return probs
    
    def select_action(self):
        """选择��作"""
        probs = self.get_action_probs(self.env.state)
        action_idx = np.random.choice(4, p=probs)
        return self.actions[action_idx], action_idx
    
    def td_update(self, state, action, next_state, reward):
        """TD(0)更新"""
        s_idx = self.state_to_idx(state)
        ns_idx = self.state_to_idx(next_state)
        
        target = reward + self.gamma * self.v[ns_idx]
        td_error = target - self.v[s_idx]
        self.v[s_idx] += self.alpha * td_error
    
    def train(self, n_episodes=500):
        """训练"""
        for _ in range(n_episodes):
            state = self.env.reset()
            done = False
            while not done:
                action, action_idx = self.select_action()
                next_state, reward, done = self.env.step(action)
                self.td_update(state, action, next_state, reward)
                state = next_state

env = GridEnvironmentWithObstacles()
agent = TDAgent(env, tau=1.0, alpha=0.1, gamma=0.9)
agent.train(n_episodes=1000)

value_grid = agent.v.reshape(env.n, env.n)
print("=== TD(0)价值函数评估结果 ===")
print("价值分布:\n", value_grid)

fig, ax = plt.subplots(figsize=(6, 6))
im = ax.imshow(value_grid, cmap='YlOrRd', origin='lower')
ax.scatter([env.obstacles[i][1] for i in range(3)],
           [env.obstacles[i][0] for i in range(3)],
           c='black', s=200, marker='X', label='障碍物')
ax.scatter(env.terminal_state[1], env.terminal_state[0],
          c='green', s=200, marker='*', label='目标')
for i in range(env.n):
    for j in range(env.n):
        if (i, j) in env.obstacles:
            ax.text(j, i, 'X', ha='center', va='center', fontsize=14, color='white')
        else:
            ax.text(j, i, f'{value_grid[i,j]:.2f}', ha='center', va='center', fontsize=10)
ax.set_title('TD(0)价值函数分布 (障碍物影响)')
ax.set_xlabel('列')
ax.set_ylabel('行')
ax.legend()
plt.colorbar(im, ax=ax, label='价值')
plt.show()

print("分析: 障碍物位置价值明显降低，智能体学习绕过障碍物的路径")

## 5. 编程题4: 增量式蒙特卡洛评估

### 题目描述
扩展任务3的蒙特卡洛评估法，实现增量式蒙特卡洛评估（每次更新价值函数时仅使用当前回合的回报，而非所有历史回报），对比增量式与普通蒙特卡洛评估的收敛速度。

In [ ]:
class MonteCarloAgent:
    """蒙特卡洛评估智能体"""
    def __init__(self, env, alpha=0.1, gamma=0.9):
        self.env = env
        self.alpha = alpha
        self.gamma = gamma
        self.n_states = env.n * env.n
        self.v = np.zeros(self.n_states)
        self.counts = np.zeros(self.n_states)
        self.actions = ['up', 'down', 'left', 'right']
    
    def state_to_idx(self, state):
        return state[0] * self.env.n + state[1]
    
    def select_action(self):
        """随机选择动作"""
        return np.random.choice(self.actions)
    
    def mc_update(self, states, rewards):
        """普通蒙特卡洛更新"""
        G = 0
        for t, (state, reward) in enumerate(zip(states, rewards)):
            G = self.gamma * G + reward
            s_idx = self.state_to_idx(state)
            self.v[s_idx] += self.alpha * (G - self.v[s_idx])
    
    def incremental_mc_update(self, state, G):
        """增量式蒙特卡洛更新"""
        s_idx = self.state_to_idx(state)
        self.counts[s_idx] += 1
        self.v[s_idx] += (G - self.v[s_idx]) / self.counts[s_idx]

class IncrementalMCAgent(MonteCarloAgent):
    """增量式蒙特卡洛智能体"""
    def mc_update(self, states, rewards):
        """使用增量式更新"""
        G = 0
        for t, (state, reward) in enumerate(zip(states, rewards)):
            G = self.gamma * G + reward
            self.incremental_mc_update(state, G)

def run_episode(env, agent, max_steps=100):
    """运行一个回合"""
    states = []
    rewards = []
    state = env.reset()
    
    for _ in range(max_steps):
        action = agent.select_action()
        next_state, reward, done = env.step(action)
        states.append(state)
        rewards.append(reward)
        state = next_state
        if done:
            break
    
    return states, rewards

env = GridEnvironmentWithObstacles()
n_episodes = 500

mc_agent = MonteCarloAgent(env, alpha=0.1, gamma=0.9)
inc_agent = IncrementalMCAgent(env, alpha=0.1, gamma=0.9)

mc_errors = []
inc_errors = []

for ep in range(n_episodes):
    mc_agent = MonteCarloAgent(env, alpha=0.1, gamma=0.9)
    states, rewards = run_episode(env, mc_agent)
    mc_agent.mc_update(states, rewards)
    mc_error = np.abs(mc_agent.v[env.state_to_idx(env.terminal_state)] - 1.0)
    mc_errors.append(mc_error)
    
    inc_agent = IncrementalMCAgent(env, alpha=0.1, gamma=0.9)
    states, rewards = run_episode(env, inc_agent)
    inc_agent.mc_update(states, rewards)
    inc_error = np.abs(inc_agent.v[env.state_to_idx(env.terminal_state)] - 1.0)
    inc_errors.append(inc_error)

plt.figure(figsize=(10, 5))
plt.plot(mc_errors, label='普通蒙特卡洛', alpha=0.7)
plt.plot(inc_errors, label='增量式蒙特卡洛', alpha=0.7)
plt.xlabel('回合数')
plt.ylabel('目标状态价值误差')
plt.title('收敛速度对比: 普通 vs 增量式蒙特卡洛')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("结论: 增量式蒙特卡洛通过记录访问次数实现更稳定的收敛")

## 6. 本章小结

本章介绍了强化学习的核心要素:
1. 智能体: 感知、决策、学习三大模块
2. 环境: 观测性、时间特性、动力学特性、任务特性
3. 交互模型: 状态、动作、奖励、策略
4. 策略评估: 蒙特卡洛、时序差分、动态规划

通过4个编程练习，掌握了:
- 带障碍物的网格世界环境实现
- Softmax策略的动作选择
- TD(0)时序差分学习
- 增量式蒙特卡洛评估